In [ ]:
"""
Feature analysis notebook for understanding anomaly detection.
"""

# Cell 1: Setup
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

# Add backend to path
sys.path.insert(0, os.path.join(os.getcwd(), "backend"))

from src.signal.features import FeatureExtractor
from src.anomaly.rules import RuleBasedDetector

print("✓ Setup complete")


# Cell 2: Generate synthetic normal and abnormal signals
def generate_normal_signal(duration=10, sampling_rate=30):
    """Generate normal vibration signal (5 Hz primary)."""
    t = np.arange(0, duration, 1/sampling_rate)
    # Primary component at 5 Hz
    signal_data = 0.3 * np.sin(2 * np.pi * 5 * t)
    # Small noise
    signal_data += 0.05 * np.random.normal(0, 1, len(signal_data))
    return signal_data


def generate_abnormal_signal(duration=10, sampling_rate=30, anomaly_type="imbalance"):
    """Generate abnormal vibration signal."""
    t = np.arange(0, duration, 1/sampling_rate)
    
    if anomaly_type == "imbalance":
        # Higher amplitude, multiple frequencies
        signal_data = 0.7 * np.sin(2 * np.pi * 5 * t)
        signal_data += 0.4 * np.sin(2 * np.pi * 10 * t)
        signal_data += 0.1 * np.random.normal(0, 1, len(signal_data))
    
    elif anomaly_type == "loosened_bolt":
        # Frequency shift
        signal_data = 0.5 * np.sin(2 * np.pi * 12 * t)
        signal_data += 0.3 * np.sin(2 * np.pi * 3 * t)
        signal_data += 0.08 * np.random.normal(0, 1, len(signal_data))
    
    elif anomaly_type == "bearing_damage":
        # High frequency spikes
        signal_data = 0.2 * np.sin(2 * np.pi * 5 * t)
        signal_data += 0.5 * np.sin(2 * np.pi * 45 * t)
        signal_data += 0.1 * np.random.normal(0, 1, len(signal_data))
    
    else:  # general degradation
        signal_data = 0.6 * np.sin(2 * np.pi * 5 * t)
        signal_data += 0.3 * np.random.normal(0, 1, len(signal_data))
    
    return signal_data


# Generate multiple samples
print("Generating synthetic data...")
num_samples = 20

normal_signals = [generate_normal_signal() for _ in range(num_samples)]
imbalance_signals = [generate_abnormal_signal(anomaly_type="imbalance") for _ in range(num_samples)]
loosened_signals = [generate_abnormal_signal(anomaly_type="loosened_bolt") for _ in range(num_samples)]
bearing_signals = [generate_abnormal_signal(anomaly_type="bearing_damage") for _ in range(num_samples)]

print(f"✓ Generated {num_samples} normal and {num_samples*3} abnormal signals")


# Cell 3: Extract features for all signals
print("Extracting features...")

feature_extractor = FeatureExtractor(sampling_rate=30)

def extract_all_features(signal_list):
    """Extract features from list of signals."""
    features_list = []
    for sig in signal_list:
        features = feature_extractor.extract_features(sig)
        features_list.append(features)
    return features_list


normal_features = extract_all_features(normal_signals)
imbalance_features = extract_all_features(imbalance_signals)
loosened_features = extract_all_features(loosened_signals)
bearing_features = extract_all_features(bearing_signals)

print("✓ Feature extraction complete")


# Cell 4: Feature comparison
print("\nFeature Statistics Comparison:")
print("="*80)

features_by_class = {
    "Normal": normal_features,
    "Imbalance": imbalance_features,
    "Loosened": loosened_features,
    "Bearing": bearing_features,
}

comparison_features = ["rms", "dominant_frequency", "variance", "spectral_entropy"]

for feat_name in comparison_features:
    print(f"\n{feat_name}:")
    print("-" * 60)
    
    for class_name, feat_list in features_by_class.items():
        values = [f[feat_name] for f in feat_list]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"  {class_name:12s}: mean={mean_val:10.4f}, std={std_val:10.4f}")


# Cell 5: Visualize feature distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Feature Distributions by Fault Type")

for idx, feat_name in enumerate(comparison_features):
    ax = axes[idx // 2, idx % 2]
    
    for class_name, feat_list in features_by_class.items():
        values = [f[feat_name] for f in feat_list]
        ax.hist(values, alpha=0.5, label=class_name, bins=8)
    
    ax.set_title(feat_name)
    ax.set_ylabel("Frequency")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Feature distribution plots complete")


# Cell 6: Anomaly detection performance
print("Testing Rule-Based Anomaly Detector...")
print("="*80)

detector = RuleBasedDetector()

# Train baseline on normal data
detector.update_baseline(normal_features)

# Test on all classes
all_features = {
    "Normal": normal_features,
    "Imbalance": imbalance_features,
    "Loosened": loosened_features,
    "Bearing": bearing_features,
}

results_summary = {}

for class_name, feat_list in all_features.items():
    indices = []
    statuses = []
    
    for feat in feat_list:
        status, index = detector.detect(feat)
        indices.append(index)
        statuses.append(1 if status == "Normal" else 0)
    
    mean_index = np.mean(indices)
    normal_count = sum(statuses)
    total = len(statuses)
    
    results_summary[class_name] = {
        "mean_index": mean_index,
        "detected_normal": normal_count,
        "total": total,
        "detection_rate": normal_count / total if total > 0 else 0
    }
    
    print(f"\n{class_name}:")
    print(f"  Mean Anomaly Index: {mean_index:6.3f}")
    print(f"  Detected as Normal: {normal_count:2d}/{total:2d} ({normal_count/total*100:5.1f}%)")


# Cell 7: Visualize detection results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of anomaly indices
class_names = list(results_summary.keys())
indices = [results_summary[c]["mean_index"] for c in class_names]
colors = ['green' if v > 0.6 else 'red' for v in indices]

axes[0].bar(class_names, indices, color=colors, alpha=0.7)
axes[0].axhline(y=0.6, color='orange', linestyle='--', label='Normal Threshold')
axes[0].set_ylabel('Anomaly Index')
axes[0].set_title('Detection Results - Anomaly Index')
axes[0].legend()
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# Detection accuracy
detections = [results_summary[c]["detected_normal"] / results_summary[c]["total"] 
              for c in class_names]
axes[1].bar(class_names, detections, color=colors, alpha=0.7)
axes[1].set_ylabel('Detection Rate')
axes[1].set_title('Detection Rate - "Normal" Classification')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Analysis complete")


# Cell 8: Recommendations
print("\n" + "="*80)
print("RECOMMENDATIONS FOR THRESHOLD TUNING")
print("="*80)

print("\nCurrent detections (threshold = 0.6):")
for class_name, metrics in results_summary.items():
    if class_name == "Normal":
        print(f"  {class_name}: {metrics['detection_rate']*100:5.1f}% correct")
    else:
        print(f"  {class_name}: {(1-metrics['detection_rate'])*100:5.1f}% detected as anomaly")

print("\nTo improve detection:")
print("  1. Lower threshold (0.5) → more sensitive to anomalies")
print("  2. Raise threshold (0.7) → more conservative detection")
print("  3. Collect real machine data for training/validation")
print("  4. Consider ML-based classification (SVM, Random Forest)")